# 3D U-Net on BraTS2020 — resumable Colab training

**Why this notebook exists.** Our hackathon models are 2D: we segment slice by
slice, because a 3D BraTS model documents a 16 GB+ VRAM requirement and the
laptop has 6 GB. That is a hardware limit, not a method limit, and it left an
open question — *how much does 3D actually buy?*

This answers it. A Colab T4 has ~15 GB, which is enough for patch-based 3D
training, so we can measure 3D against our 2D result on the **same patient-level
splits with the same metrics** and get a number instead of a hypothesis.

**Baseline to beat:** mean tumour Dice **0.76**, enhancing tumour **0.84**.

---

### How the resuming works

Colab disconnects — on idle, on timeout, or at random. So:

* every epoch writes a checkpoint to **Google Drive**, not to the VM disk
* on start, the notebook looks for the newest checkpoint and continues from it
* the metric history travels inside the checkpoint, so the curves survive too

That means you can close the tab, come back tomorrow, run all cells, and it
picks up where it stopped. Nothing here needs to finish in one session.

**Run order:** cells 1 → 8 the first time. Every time after that, run all cells
again; cell 7 detects the checkpoint and resumes automatically.

## 1 · Check the GPU

Colab assigns different cards. Anything with **15 GB or more** runs the settings
below unchanged. If you get a smaller card, drop `PATCH` to 96 in cell 4.

*Runtime → Change runtime type → T4 GPU* if this reports no GPU.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

import torch
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0),
          f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## 2 · Mount Drive

Checkpoints go here. This is the whole reason the run survives a disconnect —
anything written to the VM's own disk is gone when the session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CKPT_DIR = '/content/drive/MyDrive/mri_3d_unet'
os.makedirs(CKPT_DIR, exist_ok=True)
print("checkpoints ->", CKPT_DIR)
print("existing:", sorted(os.listdir(CKPT_DIR)) or "(none yet — first run)")

## 3 · Get BraTS2020

**Do not upload the dataset from your laptop** — it is 14 GB and you would be
uploading over a home connection. Pull it straight into Colab instead, which
runs at datacentre speed and takes a few minutes.

You need a Kaggle API token: kaggle.com → your profile → Settings → API →
*Create New Token*. That downloads `kaggle.json`. Upload it when prompted.

In [ ]:
import os

if not os.path.exists('/content/data/MICCAI_BraTS2020_TrainingData'):
    from google.colab import files
    if not os.path.exists('/root/.kaggle/kaggle.json'):
        print("Upload kaggle.json:")
        up = files.upload()
        os.makedirs('/root/.kaggle', exist_ok=True)
        os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
        os.chmod('/root/.kaggle/kaggle.json', 0o600)

    !pip install -q kaggle
    !kaggle datasets download -d awsaf49/brats20-dataset-training-validation -p /content --unzip -q
    !mkdir -p /content/data
    !mv /content/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData /content/data/ 2>/dev/null || true

ROOT = '/content/data/MICCAI_BraTS2020_TrainingData'
cases = sorted(d for d in os.listdir(ROOT) if d.startswith('BraTS20'))
print(f"{len(cases)} cases available")

## 4 · Configuration

`PATCH = 128` means we train on random 128×128×128 crops rather than whole
240×240×155 volumes. Full volumes will not fit, and patches are the standard
approach for 3D medical segmentation anyway — they also act as augmentation,
since each epoch sees different crops.

**Splitting is at patient level, never at patch or slice level.** Patches from
one patient are highly correlated; splitting below the patient would leak
information between train and validation and inflate every score. This is the
same rule the 2D work used, which is what makes the comparison fair.

In [ ]:
PATCH       = 128     # drop to 96 if you got a smaller GPU
BATCH       = 1       # 3D patches are large; accumulate instead of batching
ACCUM       = 2       # effective batch = BATCH * ACCUM
BASE_FILT   = 16      # 32 doubles memory; 16 fits comfortably at 128^3
LR          = 1e-3
EPOCHS      = 60      # total across ALL sessions, not per session
N_CASES     = 126     # match the 2D run so the comparison is like for like
VAL_FRAC    = 0.2
SEED        = 42

# BraTS labels: 0 bg, 1 necrotic, 2 oedema, 4 enhancing.
# Label 3 does not exist in the raw data, so 4 is remapped to 3 to keep the
# classes contiguous — PyTorch's CrossEntropyLoss requires that.
NUM_CLASSES = 4
MODALITIES  = ['t1', 't1ce', 't2', 'flair']

import random, numpy as np, torch
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f"patch {PATCH}^3 | effective batch {BATCH*ACCUM} | {EPOCHS} epochs total")

## 5 · Dataset

Each item is a random patch from one patient, with all four modalities stacked
as channels — the same 4-channel input the 2D model used, for the same reason:
each sequence reveals a different part of the tumour (T1c the enhancing rim,
FLAIR/T2 the oedema).

Patches are biased towards tumour-containing regions. Without that bias, most
random 128³ crops from a 240×240×155 volume contain no tumour at all, and the
model spends the run learning to predict background.

In [ ]:
import numpy as np, nibabel as nib, torch
from torch.utils.data import Dataset, DataLoader

def norm(v):
    b = v[v > 0]
    if b.size == 0:
        return v.astype(np.float32)
    return np.clip((v - b.mean()) / (b.std() + 1e-8), -5, 5).astype(np.float32)

class BraTS3D(Dataset):
    def __init__(self, case_dirs, patch=PATCH, train=True, samples=2):
        self.dirs, self.patch, self.train = case_dirs, patch, train
        self.samples = samples if train else 1

    def __len__(self):
        return len(self.dirs) * self.samples

    def __getitem__(self, i):
        d = self.dirs[i // self.samples]
        cid = os.path.basename(d)
        vols = [norm(nib.load(f"{d}/{cid}_{m}.nii").get_fdata()) for m in MODALITIES]
        seg = nib.load(f"{d}/{cid}_seg.nii").get_fdata().astype(np.int64)
        seg[seg == 4] = 3                                   # contiguous classes
        x = np.stack(vols)                                   # 4,H,W,D

        p = self.patch
        if self.train:
            # bias the crop towards tumour: a uniform random crop usually
            # contains none, and the model then learns only background
            fg = np.argwhere(seg > 0)
            if len(fg) and np.random.rand() < 0.7:
                c = fg[np.random.randint(len(fg))]
                st = [int(np.clip(c[k] - p // 2, 0, max(seg.shape[k] - p, 0)))
                      for k in range(3)]
            else:
                st = [np.random.randint(0, max(seg.shape[k] - p, 1)) for k in range(3)]
        else:
            st = [max((seg.shape[k] - p) // 2, 0) for k in range(3)]

        sl = tuple(slice(st[k], st[k] + p) for k in range(3))
        x, y = x[(slice(None),) + sl], seg[sl]

        # pad if the volume was smaller than the patch on any axis
        pad = [(0, max(p - x.shape[k + 1], 0)) for k in range(3)]
        if any(b for _a, b in pad):
            x = np.pad(x, [(0, 0)] + pad)
            y = np.pad(y, pad)
        return torch.from_numpy(x.copy()), torch.from_numpy(y.copy())

dirs = [os.path.join(ROOT, c) for c in cases[:N_CASES]]
dirs = [d for d in dirs if os.path.exists(f"{d}/{os.path.basename(d)}_seg.nii")]
random.Random(SEED).shuffle(dirs)                            # PATIENT-level split
n_val = max(1, int(len(dirs) * VAL_FRAC))
val_dirs, train_dirs = dirs[:n_val], dirs[n_val:]

train_dl = DataLoader(BraTS3D(train_dirs, train=True), batch_size=BATCH,
                      shuffle=True, num_workers=2, pin_memory=True)
val_dl = DataLoader(BraTS3D(val_dirs, train=False), batch_size=1, num_workers=2)
print(f"{len(train_dirs)} train patients / {len(val_dirs)} val patients "
      f"({len(train_dl)} train patches per epoch)")

## 6 · Model and loss

**Loss = Cross-Entropy + soft Dice**, identical to the 2D run.

Dice is not optional here. Background is **99.03 %** of voxels in BraTS, so a
model trained on cross-entropy alone reaches 99 % accuracy by predicting
"background" everywhere and finding no tumour at all. Dice measures region
overlap, so that degenerate solution scores zero.

In [ ]:
import torch.nn as nn, torch.nn.functional as F

def block(i, o):
    return nn.Sequential(
        nn.Conv3d(i, o, 3, padding=1, bias=False), nn.InstanceNorm3d(o), nn.LeakyReLU(0.01, True),
        nn.Conv3d(o, o, 3, padding=1, bias=False), nn.InstanceNorm3d(o), nn.LeakyReLU(0.01, True))

class UNet3D(nn.Module):
    """Same shape as our 2D model: encoder, bottleneck, decoder, skip
    connections. Skips carry fine boundary detail straight across, which is
    what keeps tumour edges sharp — in medicine the boundary is the finding."""
    def __init__(self, in_ch=4, n_cls=NUM_CLASSES, f=BASE_FILT):
        super().__init__()
        self.e1, self.e2, self.e3 = block(in_ch, f), block(f, f*2), block(f*2, f*4)
        self.bott = block(f*4, f*8)
        self.u3 = nn.ConvTranspose3d(f*8, f*4, 2, 2); self.d3 = block(f*8, f*4)
        self.u2 = nn.ConvTranspose3d(f*4, f*2, 2, 2); self.d2 = block(f*4, f*2)
        self.u1 = nn.ConvTranspose3d(f*2, f,   2, 2); self.d1 = block(f*2, f)
        self.out = nn.Conv3d(f, n_cls, 1)
        self.pool = nn.MaxPool3d(2)

    def forward(self, x):
        e1 = self.e1(x);  e2 = self.e2(self.pool(e1)); e3 = self.e3(self.pool(e2))
        b  = self.bott(self.pool(e3))
        d3 = self.d3(torch.cat([self.u3(b),  e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return self.out(d1)                                  # raw logits

def dice_loss(logits, target, eps=1.0):
    p = F.softmax(logits, 1)
    t = F.one_hot(target, NUM_CLASSES).permute(0, 4, 1, 2, 3).float()
    dims = (0, 2, 3, 4)
    inter = (p * t).sum(dims)
    denom = p.sum(dims) + t.sum(dims)
    return 1 - ((2 * inter + eps) / (denom + eps))[1:].mean()   # ignore background

ce = nn.CrossEntropyLoss()
def criterion(logits, target):
    return ce(logits, target) + dice_loss(logits, target)

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
model = UNet3D().to(dev)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters "
      f"(the 2D model has 7.77 M)")

## 7 · Resume from the last checkpoint

This is the cell that makes the run survive disconnects. It looks in Drive for
the newest checkpoint and restores the weights, the optimiser state, the epoch
counter and the metric history.

First run: reports "starting fresh". Every run after: reports the epoch it is
continuing from.

In [ ]:
import glob, torch

opt = torch.optim.Adam(model.parameters(), lr=LR)
scaler = torch.cuda.amp.GradScaler()
start_epoch, history, best_dice = 0, [], 0.0

ckpts = sorted(glob.glob(f"{CKPT_DIR}/epoch_*.pt"))
if ckpts:
    latest = ckpts[-1]
    ck = torch.load(latest, map_location=dev)
    model.load_state_dict(ck['model'])
    opt.load_state_dict(ck['opt'])
    if 'scaler' in ck:
        scaler.load_state_dict(ck['scaler'])
    start_epoch = ck['epoch'] + 1
    history     = ck.get('history', [])
    best_dice   = ck.get('best_dice', 0.0)
    print(f"RESUMING from {os.path.basename(latest)} -> epoch {start_epoch}")
    print(f"best mean tumour Dice so far: {best_dice:.4f}")
else:
    print("starting fresh (no checkpoint in Drive)")

print(f"will train epochs {start_epoch} -> {EPOCHS-1}")

## 8 · Train

Safe to interrupt at any point — the previous epoch's checkpoint is already in
Drive. Re-run cells 1–8 to continue.

Per-class Dice is computed on validation patients every epoch, so you can watch
the enhancing-tumour class specifically. That is the clinically important one
and the one our 2D model scores best on (0.84).

In [ ]:
import time, json, numpy as np

CLASSES = ['necrotic', 'oedema', 'enhancing']

@torch.no_grad()
def validate():
    model.eval()
    inter = np.zeros(NUM_CLASSES); denom = np.zeros(NUM_CLASSES); losses = []
    for x, y in val_dl:
        x, y = x.to(dev), y.to(dev)
        with torch.cuda.amp.autocast():
            lg = model(x)
            losses.append(criterion(lg, y).item())
        pr = lg.argmax(1)
        for c in range(NUM_CLASSES):
            pc, tc = (pr == c), (y == c)
            inter[c] += (pc & tc).sum().item()
            denom[c] += pc.sum().item() + tc.sum().item()
    dice = np.where(denom > 0, 2 * inter / np.maximum(denom, 1), np.nan)
    return float(np.mean(losses)), dice

for ep in range(start_epoch, EPOCHS):
    model.train(); t0 = time.time(); tot = 0.0
    opt.zero_grad(set_to_none=True)
    for i, (x, y) in enumerate(train_dl):
        x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
        with torch.cuda.amp.autocast():
            loss = criterion(model(x), y) / ACCUM
        scaler.scale(loss).backward()
        if (i + 1) % ACCUM == 0:
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        tot += loss.item() * ACCUM

    tr = tot / max(len(train_dl), 1)
    vl, dice = validate()
    mean_tumour = float(np.nanmean(dice[1:]))
    history.append({'epoch': ep, 'train_loss': tr, 'val_loss': vl,
                    'dice': [None if np.isnan(d) else float(d) for d in dice],
                    'mean_tumour_dice': mean_tumour})

    per = "  ".join(f"{n} {dice[i+1]:.3f}" for i, n in enumerate(CLASSES))
    print(f"epoch {ep:3d} | train {tr:.4f} | val {vl:.4f} | "
          f"mean tumour Dice {mean_tumour:.4f} | {per} | {time.time()-t0:.0f}s")

    # checkpoint EVERY epoch, to Drive — this is what survives a disconnect
    ck = {'model': model.state_dict(), 'opt': opt.state_dict(),
          'scaler': scaler.state_dict(), 'epoch': ep,
          'history': history, 'best_dice': max(best_dice, mean_tumour),
          'config': {'patch': PATCH, 'base_filters': BASE_FILT,
                     'n_cases': N_CASES, 'seed': SEED}}
    torch.save(ck, f"{CKPT_DIR}/epoch_{ep:03d}.pt")
    if mean_tumour > best_dice:
        best_dice = mean_tumour
        torch.save(ck, f"{CKPT_DIR}/best.pt")
        print(f"   new best ({best_dice:.4f}) -> best.pt")

    # keep only the 3 most recent epoch files; Drive fills up fast otherwise
    old = sorted(glob.glob(f"{CKPT_DIR}/epoch_*.pt"))[:-3]
    for f in old:
        os.remove(f)

    with open(f"{CKPT_DIR}/history.json", 'w') as f:
        json.dump(history, f, indent=2)

print(f"\ndone through epoch {EPOCHS-1}. best mean tumour Dice: {best_dice:.4f}")

## 9 · Did 3D beat 2D?

The comparison this notebook exists to make. Same dataset, same patient-level
splitting rule, same loss, same metric.

**Be careful about one thing when reading this.** Our 2D number is computed over
whole validation slices, while this validates on centre patches. That is not
exactly the same evaluation, so treat a small difference as noise. A real win
should be several points, not a fraction of one — and if 3D does *not* clearly
win, that is a legitimate result worth reporting too.

In [ ]:
import json, numpy as np
import matplotlib.pyplot as plt

hist = json.load(open(f"{CKPT_DIR}/history.json"))
ep = [h['epoch'] for h in hist]

fig, (a, b) = plt.subplots(1, 2, figsize=(13, 4.5))
a.plot(ep, [h['train_loss'] for h in hist], label='train')
a.plot(ep, [h['val_loss'] for h in hist], label='validation')
a.set_xlabel('epoch'); a.set_ylabel('loss'); a.legend()
a.set_title('Learning curve — a growing gap means overfitting')
a.grid(alpha=.3)

b.plot(ep, [h['mean_tumour_dice'] for h in hist], lw=2, label='3D U-Net (this run)')
b.axhline(0.76, ls='--', c='#b4432c', label='our 2D baseline (0.76)')
for i, n in enumerate(['necrotic', 'oedema', 'enhancing']):
    b.plot(ep, [h['dice'][i+1] for h in hist], alpha=.5, lw=1, label=n)
b.set_xlabel('epoch'); b.set_ylabel('Dice'); b.legend(fontsize=8)
b.set_title('Tumour Dice against the 2D result'); b.grid(alpha=.3)
plt.tight_layout(); plt.show()

best = max(h['mean_tumour_dice'] for h in hist)
print(f"best 3D mean tumour Dice : {best:.4f}")
print(f"our 2D baseline          : 0.7600")
d = best - 0.76
print(f"difference               : {d:+.4f}  "
      f"({'3D wins' if d > 0.02 else 'no clear win — report it as such'})")

## 10 · Bring the weights home

`best.pt` is already in your Drive. Download it if you want it in the repo —
but note the repo's `.gitignore` excludes `*.pt`, so add it to a release rather
than committing it.

**If you write this up**, state honestly:

* the evaluation is patch-based here and slice-based in the 2D run, so it is
  close but not identical
* how many epochs it actually ran (a 3D model needs longer than 2D to converge)
* the number, whichever direction it went

In [ ]:
from google.colab import files
print("size:", os.path.getsize(f"{CKPT_DIR}/best.pt") / 1e6, "MB")
files.download(f"{CKPT_DIR}/best.pt")